# 6.33 — Double Descent & Grokking

Double descent and grokking explain two surprises in modern overparameterized learning: test error can rise near the interpolation threshold and then fall again, and a model can memorize training data long before it discovers the simple rule that generalizes. In this lesson, we build the curves from scratch with NumPy so capacity, optimization, variance, and delayed generalization are visible as numbers rather than slogans.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build double descent and grokking one idea at a time. Run each cell in order and read the printed intermediate values — every piece of math is spelled out so the interpolation peak and the delayed generalization phase are not black boxes. This walkthrough is self-contained and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, polynomial features, least-squares solves, and simulations.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for data, label noise, and gradient descent.

### 1. Fitting through noisy data: the interpolation threshold

The interpolation threshold is the capacity point where a model first has enough freedom to drive training error to zero. To see it from scratch, we make noisy samples from a smooth function and fit polynomial models of increasing degree. A degree-1 line underfits, a medium polynomial starts following the curve, and a very high degree can pass through almost every noisy point.

In [ ]:
n_w = 18  # number of observed training points.
x_train_w = np.linspace(-1, 1, n_w)  # evenly spaced inputs make the geometry easy to inspect.
y_clean_w = np.sin(3 * x_train_w)  # hidden smooth rule we wish the model would learn.
noise_w = 0.18 * np.random.randn(n_w)  # small measurement noise that should not be memorized.
y_train_w = y_clean_w + noise_w  # observed labels mix signal and noise.
print("first x values:", np.round(x_train_w[:5], 3))  # inspect the training inputs.
print("first noisy y values:", np.round(y_train_w[:5], 3))  # inspect the labels the model actually sees.

▶ What you'll see: a small one-dimensional training set whose labels are a smooth sine wave plus random wiggles.

In [ ]:
def poly_design_w(x, degree):  # build a Vandermonde design matrix [1, x, x^2, ...].
    return np.vstack([x ** k for k in range(degree + 1)]).T  # rows are examples, columns are polynomial features.

def fit_poly_w(x, y, degree):  # fit the minimum-norm least-squares polynomial.
    Phi = poly_design_w(x, degree)  # feature matrix for the requested capacity.
    coef = np.linalg.pinv(Phi) @ y  # pseudoinverse handles under-, exact-, and over-parameterized cases.
    return coef  # one coefficient per feature.

coef3_w = fit_poly_w(x_train_w, y_train_w, 3)  # fit a low-capacity cubic model.
print("degree 3 parameter count:", len(coef3_w))  # degree d has d+1 parameters.
print("degree 3 coefficients:", np.round(coef3_w, 3))  # inspect the fitted curve.

▶ What you'll see: a cubic has only 4 parameters, far fewer than the 18 training points, so it cannot memorize every label.

In [ ]:
x_grid_w = np.linspace(-1.15, 1.15, 300)  # dense inputs for plotting curves between data points.
degrees_show_w = [1, 5, 17]  # underfit, moderate, and near-interpolating capacities.
plt.figure(figsize=(6, 3.6))  # compact comparison plot.
plt.scatter(x_train_w, y_train_w, color="black", s=25, label="noisy train")  # show the data to be fit.
plt.plot(x_grid_w, np.sin(3 * x_grid_w), color="gray", linestyle="--", label="clean rule")  # hidden target.
for d_w in degrees_show_w:  # draw one fitted polynomial per capacity.
    c_w = fit_poly_w(x_train_w, y_train_w, d_w)  # fit with d+1 parameters.
    yhat_w = poly_design_w(x_grid_w, d_w) @ c_w  # predict the whole grid.
    plt.plot(x_grid_w, yhat_w, label=f"degree {d_w}")  # compare curves.
plt.ylim(-2.2, 2.2); plt.title("1: capacity changes what the model can fit")  # stable axis.
plt.legend(); plt.show()  # display the curve family.

▶ What you'll see: the line underfits, the moderate polynomial tracks the smooth rule, and the degree-17 curve bends hard to hit noisy points.

*Why it's done this way:* A polynomial of degree `d` has `d+1` adjustable weights. When `d+1` approaches the number of training samples, the linear system has enough freedom to match noisy labels exactly. That point is the interpolation threshold: training error can vanish, but the fitted function may be extremely wiggly between points because it spent capacity on noise.

### 2. The double-descent curve: bias, variance, and optimization effects

Classical intuition says more capacity should eventually overfit. Double descent adds a modern twist: after the interpolation peak, even larger models can improve again because the minimum-norm solution can spread the fit across many parameters instead of using one brittle, high-variance interpolator. We can simulate this by sweeping polynomial degree and measuring train versus test error.

In [ ]:
x_test_w = np.linspace(-1, 1, 400)  # dense test inputs from the same domain.
y_test_w = np.sin(3 * x_test_w)  # evaluate against the clean rule, not the noisy labels.
degrees_w = np.arange(1, 36)  # capacity sweep from tiny to strongly overparameterized.
train_mse_w, test_mse_w, norm_w = [], [], []  # store diagnostics for every degree.
print("interpolation threshold near parameters = samples:", n_w)  # d+1 around n is the danger zone.

▶ What you'll see: for 18 training points, degree 17 has 18 parameters and is the interpolation threshold.

In [ ]:
for d_w in degrees_w:  # sweep capacities.
    c_w = fit_poly_w(x_train_w, y_train_w, int(d_w))  # minimum-norm least-squares fit.
    pred_train_w = poly_design_w(x_train_w, int(d_w)) @ c_w  # fitted values on training data.
    pred_test_w = poly_design_w(x_test_w, int(d_w)) @ c_w  # predictions on clean test grid.
    train_mse_w.append(float(np.mean((pred_train_w - y_train_w) ** 2)))  # noisy training error.
    test_mse_w.append(float(np.mean((pred_test_w - y_test_w) ** 2)))  # clean-rule generalization error.
    norm_w.append(float(np.linalg.norm(c_w)))  # coefficient size as a rough variance proxy.
train_mse_w, test_mse_w, norm_w = np.array(train_mse_w), np.array(test_mse_w), np.array(norm_w)  # arrays for plotting.
print("degree with smallest train error:", int(degrees_w[np.argmin(train_mse_w)]))  # usually at/after interpolation.
print("largest test error degree:", int(degrees_w[np.argmax(test_mse_w)]))  # often near the interpolation region.
assert train_mse_w[-1] < train_mse_w[0]  # high capacity fits training data better than low capacity.

▶ What you'll see: training error keeps falling, while the worst test error appears around a high-variance capacity.

In [ ]:
plt.figure(figsize=(6, 3.6))  # create the double-descent diagnostic plot.
plt.semilogy(degrees_w + 1, train_mse_w + 1e-12, marker="o", label="train MSE")  # parameters on x-axis.
plt.semilogy(degrees_w + 1, test_mse_w + 1e-12, marker="o", label="test MSE")  # test curve can peak.
plt.axvline(n_w, color="red", linestyle="--", label="interpolation threshold")  # p≈n.
plt.xlabel("number of polynomial parameters")  # capacity scale.
plt.ylabel("MSE, log scale")  # log scale reveals tiny training errors and large peaks.
plt.title("2: test error can rise then fall again")  # double-descent shape.
plt.legend(); plt.show()  # display the curves.

▶ What you'll see: training error decreases, but test error can spike near interpolation before improving at larger capacity.

*Why it's done this way:* The test error can be read as `bias + variance + optimization effects`. Low capacity has high bias because the function class is too rigid. Near interpolation, variance is high because tiny label noise can force large coefficient changes. In the overparameterized regime, the pseudoinverse chooses the minimum-norm interpolating solution, an implicit regularizer that can lower variance and produce the second descent.

### 3. Implicit regularization: many solutions, one simpler choice

Once there are more parameters than equations, infinitely many weight vectors can interpolate the same training labels. The pseudoinverse does not choose arbitrarily; it returns the solution with smallest Euclidean norm. That is implicit regularization: no explicit penalty was written, but the solver still prefers a simpler parameter vector.

In [ ]:
x_small_w = np.array([-1.0, -0.2, 0.35, 0.9])  # only four training inputs.
y_small_w = np.array([-1.0, -0.1, 0.4, 0.95])  # four targets to interpolate.
Phi_small_w = poly_design_w(x_small_w, 7)  # eight parameters for four equations: underdetermined.
coef_min_w = np.linalg.pinv(Phi_small_w) @ y_small_w  # minimum-norm interpolating coefficients.
resid_min_w = Phi_small_w @ coef_min_w - y_small_w  # training residuals.
print("design shape:", Phi_small_w.shape)  # 4 equations, 8 unknowns.
print("max interpolation residual:", round(float(np.max(np.abs(resid_min_w))), 10))  # should be numerical zero.
assert np.max(np.abs(resid_min_w)) < 1e-10  # verify interpolation.

▶ What you'll see: four data constraints are satisfied almost exactly even though there are eight parameters.

In [ ]:
U_w, s_s_w, Vt_s_w = np.linalg.svd(Phi_small_w, full_matrices=True)  # SVD exposes null-space directions.
rank_s_w = int(np.sum(s_s_w > 1e-10))  # numerical matrix rank.
null_vec_w = Vt_s_w[rank_s_w:].T[:, 0]  # a direction that changes coefficients but not predictions on training points.
print("rank:", rank_s_w, "null vector norm:", round(float(np.linalg.norm(null_vec_w)), 3))  # inspect null space.
print("training change from null vector:", round(float(np.linalg.norm(Phi_small_w @ null_vec_w)), 10))  # should be zero.
assert np.linalg.norm(Phi_small_w @ null_vec_w) < 1e-10  # null directions leave training predictions unchanged.

▶ What you'll see: there is a real coefficient direction that changes the model but leaves all training labels untouched.

In [ ]:
coef_wiggly_w = coef_min_w + 15 * null_vec_w  # another exact interpolator with a much larger norm.
print("min-norm coefficient norm:", round(float(np.linalg.norm(coef_min_w)), 3))  # simple solution.
print("wiggly coefficient norm:", round(float(np.linalg.norm(coef_wiggly_w)), 3))  # larger-norm solution.
plt.figure(figsize=(6, 3.4))  # compare two exact interpolators.
plt.scatter(x_small_w, y_small_w, color="black", label="training points")  # constraints both curves satisfy.
plt.plot(x_grid_w, poly_design_w(x_grid_w, 7) @ coef_min_w, label="minimum-norm interpolator")  # implicit choice.
plt.plot(x_grid_w, poly_design_w(x_grid_w, 7) @ coef_wiggly_w, label="larger-norm interpolator", linestyle="--")  # same train fit.
plt.ylim(-3, 3); plt.title("3: many interpolators, different simplicity")  # stable visual scale.
plt.legend(); plt.show()  # display both curves.

▶ What you'll see: both curves hit the same training points, but the larger-norm solution wiggles more between them.

*Why it's done this way:* Overparameterization does not mean every interpolating solution is equally good. The null-space vector proves that training data cannot distinguish infinitely many fits. A solver, architecture, optimizer, or initialization can impose a hidden preference; when that preference selects smoother or smaller-norm functions, the overparameterized model can generalize after interpolation.

### 4. Grokking: memorization first, rule later

Grokking is delayed generalization: the training objective becomes easy before the test rule becomes obvious. We can model the phenomenon with a toy modular-addition task. First a lookup table memorizes observed pairs perfectly but has no rule for held-out pairs. Then a simple rule-based representation gradually improves test accuracy while training accuracy was already high.

In [ ]:
mod_w = 7  # modular arithmetic size.
pairs_w = np.array([(a, b) for a in range(mod_w) for b in range(mod_w)])  # all ordered input pairs.
labels_w = (pairs_w[:, 0] + pairs_w[:, 1]) % mod_w  # hidden rule: addition modulo 7.
train_mask_w = (pairs_w[:, 0] < 5)  # deliberately biased training split leaves some first operands unseen.
test_mask_w = ~train_mask_w  # held-out operand values test whether a rule was learned.
print("train pairs:", int(train_mask_w.sum()), "test pairs:", int(test_mask_w.sum()))  # inspect split sizes.
print("example rule: (5 + 6) mod 7 =", int((5 + 6) % mod_w))  # held-out kind of input.

▶ What you'll see: many pairs are memorized during training, but pairs with first operand 5 or 6 require extrapolating the modular rule.

In [ ]:
lookup_pred_w = np.full(len(labels_w), -1)  # -1 means the memorizer has no answer.
lookup_pred_w[train_mask_w] = labels_w[train_mask_w]  # memorize every training label exactly.
train_acc_lookup_w = np.mean(lookup_pred_w[train_mask_w] == labels_w[train_mask_w])  # training memorization accuracy.
test_acc_lookup_w = np.mean(lookup_pred_w[test_mask_w] == labels_w[test_mask_w])  # no rule for held-out operands.
print("lookup train accuracy:", train_acc_lookup_w)  # perfect memorization.
print("lookup test accuracy:", test_acc_lookup_w)  # zero because -1 never matches labels 0..6.
assert train_acc_lookup_w == 1.0 and test_acc_lookup_w == 0.0  # verify memorization without generalization.

▶ What you'll see: the lookup table reaches 100% train accuracy while failing every held-out pair.

In [ ]:
epochs_w = np.arange(0, 401)  # not an optimizer trace; a transparent phase model for delayed rule discovery.
train_acc_w = 1 - np.exp(-epochs_w / 25)  # memorization rises quickly.
rule_strength_w = 1 / (1 + np.exp(-(epochs_w - 230) / 28))  # rule emerges late.
test_acc_w = 0.05 + 0.93 * rule_strength_w  # test accuracy stays low until the rule is found.
print("train accuracy at epoch 80:", round(float(train_acc_w[80]), 3))  # already high.
print("test accuracy at epoch 80:", round(float(test_acc_w[80]), 3))  # still low.
print("test accuracy at epoch 320:", round(float(test_acc_w[320]), 3))  # finally high.
assert train_acc_w[80] > 0.95 and test_acc_w[80] < 0.06 and test_acc_w[320] > 0.9  # delayed generalization.

▶ What you'll see: training accuracy is essentially solved early, but test accuracy improves only much later.

In [ ]:
plt.figure(figsize=(6, 3.4))  # plot the stylized grokking trace.
plt.plot(epochs_w, train_acc_w, label="train accuracy")  # quick memorization.
plt.plot(epochs_w, test_acc_w, label="test accuracy")  # delayed rule learning.
plt.axvline(230, color="red", linestyle="--", label="rule emerges")  # grokking transition.
plt.ylim(0, 1.05); plt.xlabel("epoch")  # accuracy scale.
plt.ylabel("accuracy")  # y-axis label.
plt.title("4: grokking = delayed generalization after memorization")  # concept title.
plt.legend(); plt.show()  # display the phase transition.

▶ What you'll see: a long plateau where training is high and test is low, followed by a sudden test-accuracy rise.

*Why it's done this way:* A memorizer stores individual examples and therefore wins on the training set as soon as capacity is high enough. A rule-based solution compresses many examples into one reusable structure, but optimization may reach it only after many small updates and implicit regularization pressures. Grokking is the moment the learned representation changes from table lookup to rule compression.

## 🛠️ Setup

In [ ]:
import numpy as np # load NumPy for arrays, least-squares fits, gradients, and numerical checks.
import matplotlib.pyplot as plt # load Matplotlib for all diagnostic curves and heatmaps.
np.random.seed(0) # make all examples reproducible across notebook runs.

def poly_design(x, degree): # create polynomial features [1, x, x^2, ...] from scratch.
    x = np.asarray(x, dtype=float) # ensure predictable floating-point powers.
    return np.vstack([x ** k for k in range(degree + 1)]).T # rows are examples and columns are powers.

def mse(y_true, y_pred): # compute mean squared error without external libraries.
    y_true = np.asarray(y_true, dtype=float) # convert true values to floats.
    y_pred = np.asarray(y_pred, dtype=float) # convert predictions to floats.
    return float(np.mean((y_true - y_pred) ** 2)) # average squared residuals.

def sigmoid(z): # define a stable-enough sigmoid for small teaching arrays.
    return 1 / (1 + np.exp(-z)) # logistic squashing from real numbers into (0, 1).

## 🟢 Basics (warm-up)

### Basic 1 — Make a noisy training set

**Goal.** Create a tiny regression problem with signal and noise, because double descent is about whether capacity learns the rule or memorizes the noise. We build it in 2 steps.

In [ ]:
x_b1 = np.linspace(-1, 1, 12) # define twelve evenly spaced training inputs.
y_clean_b1 = np.sin(3 * x_b1) # define the hidden smooth target rule.
y_b1 = y_clean_b1 + 0.15 * np.random.randn(len(x_b1)) # add label noise that a flexible model could memorize.
print("x shape:", x_b1.shape, "y shape:", y_b1.shape) # inspect the dataset sizes.
print("first labels:", np.round(y_b1[:4], 3)) # inspect a few noisy observations.

▶ What you'll see: a small one-dimensional dataset where labels are close to, but not exactly on, a sine curve.

In [ ]:
plt.figure(figsize=(4, 3)) # create a compact data plot.
plt.scatter(x_b1, y_b1, color="black", label="noisy labels") # show observed training labels.
plt.plot(x_b1, y_clean_b1, color="gray", linestyle="--", label="clean rule") # show the unobserved rule for teaching.
plt.title("Basic 1: signal plus noise") # title the plot.
plt.legend() # show curve labels.
plt.show() # display the figure.

▶ What you'll see: noisy points wobble around the clean curve, which creates the temptation to overfit.

👀 Takeaway: generalization means recovering the rule, not reproducing every noisy training fluctuation.

### Basic 2 — Build polynomial features

**Goal.** Turn one input coordinate into multiple powers, because model capacity increases as we add more polynomial features. We build it in 2 steps.

In [ ]:
x_b2 = np.array([-1.0, 0.0, 0.5]) # choose three simple inputs for inspecting powers.
Phi_b2 = poly_design(x_b2, 3) # build features 1, x, x^2, and x^3.
print("feature matrix:\n", Phi_b2) # inspect every generated feature.
assert Phi_b2.shape == (3, 4) # verify three examples and four polynomial parameters.

▶ What you'll see: each row contains powers of one input, and degree 3 gives four columns.

In [ ]:
plt.figure(figsize=(4, 3)) # create a feature heatmap.
plt.imshow(Phi_b2, cmap="viridis", aspect="auto") # visualize feature values.
plt.colorbar(label="feature value") # add a numeric color scale.
plt.title("Basic 2: polynomial design matrix") # title the heatmap.
plt.xlabel("power k") # label columns as polynomial powers.
plt.ylabel("example") # label rows as examples.
plt.show() # display the heatmap.

▶ What you'll see: higher powers shrink values inside (-1, 1), while the bias column is always 1.

👀 Takeaway: the number of polynomial columns is the capacity knob in these scratch experiments.

### Basic 3 — Fit a low-capacity line

**Goal.** Fit degree 1 with least squares, because underparameterized models have high bias when the true rule is curved. We build it in 2 steps.

In [ ]:
x_b3 = np.linspace(-1, 1, 12) # recreate inputs locally for this example.
y_b3 = np.sin(3 * x_b3) + 0.15 * np.random.randn(len(x_b3)) # recreate noisy labels.
Phi_b3 = poly_design(x_b3, 1) # build a line: intercept and slope.
coef_b3 = np.linalg.pinv(Phi_b3) @ y_b3 # solve least squares from scratch with the pseudoinverse.
print("line coefficients:", np.round(coef_b3, 3)) # inspect intercept and slope.

▶ What you'll see: degree 1 has only two coefficients, so the model cannot bend with the sine wave.

In [ ]:
pred_b3 = Phi_b3 @ coef_b3 # compute fitted values on the training inputs.
err_b3 = mse(y_b3, pred_b3) # compute training mean squared error.
print("line train MSE:", round(err_b3, 3)) # inspect underfit error.
plt.figure(figsize=(4, 3)) # create a line-fit figure.
plt.scatter(x_b3, y_b3, color="black") # plot noisy observations.
plt.plot(x_b3, pred_b3, color="red", label="degree 1") # plot the fitted line.
plt.title("Basic 3: underfitting with a line") # title the plot.
plt.legend(); plt.show() # display the fit.

▶ What you'll see: the fitted line misses the curved pattern even where many points agree.

👀 Takeaway: low capacity lowers variance but creates bias when the true relationship is nonlinear.

### Basic 4 — Fit an interpolating polynomial

**Goal.** Fit a polynomial with as many parameters as training points, because this is the interpolation threshold. We build it in 3 steps.

In [ ]:
x_b4 = np.linspace(-1, 1, 12) # twelve training inputs.
y_b4 = np.sin(3 * x_b4) + 0.15 * np.random.randn(len(x_b4)) # noisy observed labels.
degree_b4 = len(x_b4) - 1 # degree 11 has 12 parameters for 12 data points.
Phi_b4 = poly_design(x_b4, degree_b4) # square interpolation design matrix.
print("design shape:", Phi_b4.shape) # inspect equations versus unknowns.
assert Phi_b4.shape == (12, 12) # verify the threshold shape.

▶ What you'll see: the design matrix is square, matching twelve constraints with twelve coefficients.

In [ ]:
coef_b4 = np.linalg.pinv(Phi_b4) @ y_b4 # solve the interpolation system.
pred_b4 = Phi_b4 @ coef_b4 # compute training predictions.
train_mse_b4 = mse(y_b4, pred_b4) # training error should be numerical zero.
print("interpolating train MSE:", train_mse_b4) # inspect near-perfect training fit.
assert train_mse_b4 < 1e-18 # verify interpolation to numerical precision.

▶ What you'll see: training MSE is essentially zero because capacity matches the number of samples.

In [ ]:
x_grid_b4 = np.linspace(-1, 1, 200) # dense plotting grid.
y_grid_b4 = poly_design(x_grid_b4, degree_b4) @ coef_b4 # evaluate the interpolating polynomial.
plt.figure(figsize=(4, 3)) # create a curve plot.
plt.scatter(x_b4, y_b4, color="black") # show training points.
plt.plot(x_grid_b4, y_grid_b4, color="purple") # show the fitted interpolator.
plt.title("Basic 4: interpolation threshold") # title the plot.
plt.show() # display the curve.

▶ What you'll see: the curve passes through all training labels, including their random noise.

👀 Takeaway: zero training error is not the same as learning the clean rule.

### Basic 5 — Measure test error on the clean rule

**Goal.** Compare training fit with clean-rule error, because double descent is a generalization story rather than a training-loss story. We build it in 3 steps.

In [ ]:
x_train_b5 = np.linspace(-1, 1, 14) # training inputs.
y_train_b5 = np.sin(3 * x_train_b5) + 0.12 * np.random.randn(len(x_train_b5)) # noisy training labels.
x_test_b5 = np.linspace(-1, 1, 200) # dense test grid.
y_test_b5 = np.sin(3 * x_test_b5) # clean test target.
print("train size:", len(x_train_b5), "test size:", len(x_test_b5)) # inspect data sizes.

▶ What you'll see: the test set is much denser and uses the clean underlying function.

In [ ]:
degree_b5 = 13 # parameter count equals fourteen training points.
coef_b5 = np.linalg.pinv(poly_design(x_train_b5, degree_b5)) @ y_train_b5 # fit near interpolation.
train_err_b5 = mse(y_train_b5, poly_design(x_train_b5, degree_b5) @ coef_b5) # noisy training MSE.
test_err_b5 = mse(y_test_b5, poly_design(x_test_b5, degree_b5) @ coef_b5) # clean test MSE.
print("train MSE:", round(train_err_b5, 8), "test MSE:", round(test_err_b5, 3)) # inspect the gap.
assert train_err_b5 < 1e-16 # verify interpolation.

▶ What you'll see: training error is zero while test error remains positive because noise was fitted.

In [ ]:
plt.figure(figsize=(4, 3)) # create a train-versus-test bar chart.
plt.bar(["train", "test"], [train_err_b5 + 1e-8, test_err_b5], color=["teal", "orange"]) # add epsilon so train bar is visible.
plt.yscale("log") # log scale shows the huge difference.
plt.title("Basic 5: zero train error can still generalize poorly") # title the plot.
plt.ylabel("MSE, log scale") # label the error axis.
plt.show() # display the comparison.

▶ What you'll see: the test bar towers over the nearly zero training bar.

👀 Takeaway: interpolation is a training milestone, not a guarantee of good test performance.

### Basic 6 — Plot a capacity sweep

**Goal.** Sweep polynomial degree and watch train error fall, because increasing capacity makes the model more able to fit the observed labels. We build it in 3 steps.

In [ ]:
x_b6 = np.linspace(-1, 1, 16) # training inputs for the sweep.
y_b6 = np.sin(3 * x_b6) + 0.10 * np.random.randn(len(x_b6)) # noisy labels.
degrees_b6 = np.arange(1, 22) # degree sweep.
train_errors_b6 = [] # collect training MSE values.
print("interpolation degree:", len(x_b6) - 1) # degree 15 has 16 parameters.

▶ What you'll see: the interpolation degree is one less than the number of training samples.

In [ ]:
for degree_b6 in degrees_b6: # fit one model per degree.
    coef_b6 = np.linalg.pinv(poly_design(x_b6, int(degree_b6))) @ y_b6 # solve least squares.
    pred_b6 = poly_design(x_b6, int(degree_b6)) @ coef_b6 # training predictions.
    train_errors_b6.append(mse(y_b6, pred_b6)) # store training error.
train_errors_b6 = np.array(train_errors_b6) # convert to array for checks and plotting.
print("first train MSE:", round(float(train_errors_b6[0]), 3), "last train MSE:", round(float(train_errors_b6[-1]), 8)) # inspect decrease.
assert train_errors_b6[-1] < train_errors_b6[0] # verify capacity lowers training error.

▶ What you'll see: high-degree models fit the training labels much more closely than low-degree models.

In [ ]:
plt.figure(figsize=(4, 3)) # create the sweep plot.
plt.semilogy(degrees_b6 + 1, train_errors_b6 + 1e-12, marker="o", color="teal") # plot training MSE against parameter count.
plt.axvline(len(x_b6), color="red", linestyle="--") # mark interpolation threshold.
plt.title("Basic 6: training error falls with capacity") # title the plot.
plt.xlabel("parameters") # label capacity axis.
plt.ylabel("train MSE, log scale") # label error axis.
plt.show() # display the curve.

▶ What you'll see: train error drops sharply as the parameter count approaches the number of data points.

👀 Takeaway: the first descent is mostly about reducing bias and fitting the training set.

### Basic 7 — Locate the interpolation threshold

**Goal.** Compute where parameters match samples, because double descent is organized around that threshold. We build it in 2 steps.

In [ ]:
n_samples_b7 = 20 # choose a training-set size.
degrees_b7 = np.arange(0, 30) # candidate polynomial degrees.
params_b7 = degrees_b7 + 1 # degree d has d+1 coefficients.
threshold_degree_b7 = int(degrees_b7[np.where(params_b7 == n_samples_b7)[0][0]]) # solve d+1=n.
print("threshold degree:", threshold_degree_b7) # inspect the degree.
print("threshold parameters:", params_b7[threshold_degree_b7]) # inspect matching parameter count.
assert threshold_degree_b7 == 19 # verify d = n - 1.

▶ What you'll see: with 20 samples, the threshold is degree 19 because it has 20 parameters.

In [ ]:
plt.figure(figsize=(4, 3)) # create a parameter-count plot.
plt.plot(degrees_b7, params_b7, marker="o", color="navy") # plot parameters versus degree.
plt.axhline(n_samples_b7, color="red", linestyle="--", label="sample count") # mark data constraints.
plt.axvline(threshold_degree_b7, color="gray", linestyle="--", label="threshold degree") # mark threshold.
plt.title("Basic 7: where interpolation begins") # title the plot.
plt.xlabel("degree") # label degree axis.
plt.ylabel("parameters") # label parameter count.
plt.legend(); plt.show() # display the threshold.

▶ What you'll see: the parameter-count line crosses the sample-count line at the interpolation threshold.

👀 Takeaway: interpolation begins when capacity can satisfy every training constraint.

### Basic 8 — Watch coefficient norm as a variance proxy

**Goal.** Track coefficient size, because large coefficients often signal a wiggly high-variance interpolator. We build it in 3 steps.

In [ ]:
x_b8 = np.linspace(-1, 1, 15) # training inputs.
y_b8 = np.sin(3 * x_b8) + 0.10 * np.random.randn(len(x_b8)) # noisy labels.
degrees_b8 = np.arange(1, 23) # capacity sweep.
norms_b8 = [] # collect coefficient norms.
print("sample count:", len(x_b8)) # inspect interpolation reference.

▶ What you'll see: fifteen samples set the threshold near fifteen parameters.

In [ ]:
for degree_b8 in degrees_b8: # fit each capacity.
    coef_b8 = np.linalg.pinv(poly_design(x_b8, int(degree_b8))) @ y_b8 # minimum-norm least-squares coefficients.
    norms_b8.append(float(np.linalg.norm(coef_b8))) # store coefficient size.
norms_b8 = np.array(norms_b8) # convert to array.
print("smallest norm:", round(float(norms_b8.min()), 3), "largest norm:", round(float(norms_b8.max()), 3)) # inspect spread.
assert norms_b8.max() > norms_b8.min() # verify capacity changes coefficient size.

▶ What you'll see: some capacities require much larger coefficient vectors than others.

In [ ]:
plt.figure(figsize=(4, 3)) # create norm plot.
plt.semilogy(degrees_b8 + 1, norms_b8, marker="o", color="purple") # log scale for large changes.
plt.axvline(len(x_b8), color="red", linestyle="--") # mark interpolation threshold.
plt.title("Basic 8: coefficient norm as a warning sign") # title the plot.
plt.xlabel("parameters") # label capacity.
plt.ylabel("||w||₂, log scale") # label norm.
plt.show() # display the curve.

▶ What you'll see: coefficient norm can inflate near brittle interpolating fits.

👀 Takeaway: capacity affects not only error but also how violently the fitted function moves.

### Basic 9 — Build a memorization lookup table

**Goal.** Show perfect training accuracy without a rule, because grokking starts with memorization. We build it in 2 steps.

In [ ]:
mod_b9 = 5 # small modular arithmetic problem.
pairs_b9 = np.array([(a, b) for a in range(mod_b9) for b in range(mod_b9)]) # all input pairs.
labels_b9 = (pairs_b9[:, 0] + pairs_b9[:, 1]) % mod_b9 # hidden addition rule.
train_mask_b9 = pairs_b9[:, 0] < 4 # leave operand 4 out for testing.
print("train examples:", int(train_mask_b9.sum()), "test examples:", int((~train_mask_b9).sum())) # inspect split.

▶ What you'll see: most pairs are available for training, but an entire operand value is held out.

In [ ]:
lookup_b9 = np.full(len(labels_b9), -1) # initialize no-answer predictions.
lookup_b9[train_mask_b9] = labels_b9[train_mask_b9] # memorize every training label exactly.
train_acc_b9 = np.mean(lookup_b9[train_mask_b9] == labels_b9[train_mask_b9]) # training accuracy.
test_acc_b9 = np.mean(lookup_b9[~train_mask_b9] == labels_b9[~train_mask_b9]) # held-out accuracy.
print("lookup train acc:", train_acc_b9, "test acc:", test_acc_b9) # inspect memorization gap.
assert train_acc_b9 == 1.0 and test_acc_b9 == 0.0 # verify no rule generalization.

▶ What you'll see: memorization solves the training set and fails the held-out operand.

👀 Takeaway: high training accuracy can come from lookup-table memory rather than a learned rule.

### Basic 10 — Draw a grokking-shaped curve

**Goal.** Visualize delayed generalization, because grokking is defined by test accuracy rising long after train accuracy. We build it in 3 steps.

In [ ]:
epochs_b10 = np.arange(0, 301) # training timeline.
train_b10 = 1 - np.exp(-epochs_b10 / 25) # fast memorization curve.
test_b10 = 0.1 + 0.85 * sigmoid((epochs_b10 - 180) / 22) # delayed test improvement.
print("train at 80:", round(float(train_b10[80]), 3), "test at 80:", round(float(test_b10[80]), 3)) # inspect early gap.
assert train_b10[80] > 0.95 and test_b10[80] < 0.12 # verify memorization-before-generalization.

▶ What you'll see: training accuracy is already high when test accuracy is still near chance.

In [ ]:
print("test at 250:", round(float(test_b10[250]), 3)) # inspect late generalization.
assert test_b10[250] > 0.9 # verify delayed rule discovery.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create grokking plot.
plt.plot(epochs_b10, train_b10, label="train") # plot memorization.
plt.plot(epochs_b10, test_b10, label="test") # plot delayed generalization.
plt.axvline(180, color="red", linestyle="--", label="late transition") # mark grokking onset.
plt.title("Basic 10: grokking-shaped accuracy") # title the plot.
plt.xlabel("epoch") # label time axis.
plt.ylabel("accuracy") # label accuracy axis.
plt.ylim(0, 1.05); plt.legend(); plt.show() # display the curves.

▶ What you'll see: the test curve stays flat and then rises sharply after the training curve has saturated.

👀 Takeaway: grokking separates memorizing examples from discovering the compressed rule.

## 🟡 Easy

### Easy 1 — Reproduce a double-descent sweep

**Goal.** Measure train and clean-test MSE over capacity, because the double-descent curve is empirical evidence about generalization. We build it in 4 steps.

In [ ]:
x_e1 = np.linspace(-1, 1, 18) # choose the training inputs.
y_e1 = np.sin(3 * x_e1) + 0.18 * np.random.randn(len(x_e1)) # create noisy observations.
x_test_e1 = np.linspace(-1, 1, 300) # dense test grid.
y_test_e1 = np.sin(3 * x_test_e1) # clean target function.
degrees_e1 = np.arange(1, 36) # sweep from underfit to overparameterized.
print("threshold parameters:", len(x_e1)) # interpolation reference.

▶ What you'll see: there are 18 training constraints, so 18 parameters marks interpolation.

In [ ]:
train_e1, test_e1 = [], [] # store errors.
for degree_e1 in degrees_e1: # fit every capacity.
    coef_e1 = np.linalg.pinv(poly_design(x_e1, int(degree_e1))) @ y_e1 # minimum-norm polynomial fit.
    train_e1.append(mse(y_e1, poly_design(x_e1, int(degree_e1)) @ coef_e1)) # training MSE.
    test_e1.append(mse(y_test_e1, poly_design(x_test_e1, int(degree_e1)) @ coef_e1)) # clean test MSE.
train_e1, test_e1 = np.array(train_e1), np.array(test_e1) # arrays for analysis.
print("min train MSE:", round(float(train_e1.min()), 10), "max test MSE:", round(float(test_e1.max()), 3)) # inspect ranges.
assert train_e1[-1] < train_e1[0] # verify training improves with capacity.

In [ ]:
peak_degree_e1 = int(degrees_e1[np.argmax(test_e1)]) # locate worst generalization.
best_after_e1 = float(np.min(test_e1[degrees_e1 + 1 > len(x_e1)])) # best overparameterized error.
print("test peak degree:", peak_degree_e1, "best after threshold:", round(best_after_e1, 3)) # inspect double-descent pieces.
assert best_after_e1 < float(np.max(test_e1)) # verify there is descent after the peak.

In [ ]:
plt.figure(figsize=(5, 3)) # create double-descent plot.
plt.semilogy(degrees_e1 + 1, train_e1 + 1e-12, marker="o", label="train") # train curve.
plt.semilogy(degrees_e1 + 1, test_e1 + 1e-12, marker="o", label="test") # test curve.
plt.axvline(len(x_e1), color="red", linestyle="--", label="p≈n") # threshold.
plt.title("Easy 1: double-descent capacity sweep") # title the plot.
plt.xlabel("parameters") # label capacity.
plt.ylabel("MSE, log scale") # label error.
plt.legend(); plt.show() # display.

▶ What you'll see: the test curve can rise around interpolation and then improve again beyond it.

👀 Takeaway: double descent is about the shape of test error as capacity crosses the interpolation threshold.

### Easy 2 — Compare explicit ridge regularization

**Goal.** Add a ridge penalty to a near-interpolating polynomial, because explicit regularization can reduce the variance spike. We build it in 3 steps.

In [ ]:
x_e2 = np.linspace(-1, 1, 16) # training inputs.
y_e2 = np.sin(3 * x_e2) + 0.14 * np.random.randn(len(x_e2)) # noisy labels.
x_test_e2 = np.linspace(-1, 1, 250) # test grid.
y_test_e2 = np.sin(3 * x_test_e2) # clean target.
degree_e2 = 15 # threshold degree for sixteen points.
Phi_e2 = poly_design(x_e2, degree_e2) # threshold design matrix.
Phi_test_e2 = poly_design(x_test_e2, degree_e2) # test features.
print("Phi shape:", Phi_e2.shape) # inspect square interpolation system.

▶ What you'll see: degree 15 creates 16 parameters for 16 examples.

In [ ]:
lams_e2 = np.array([0.0, 1e-5, 1e-3, 1e-1]) # ridge strengths from none to strong.
train_e2, test_e2, norms_e2 = [], [], [] # collect diagnostics.
for lam_e2 in lams_e2: # solve ridge for each lambda.
    A_e2 = Phi_e2.T @ Phi_e2 + lam_e2 * np.eye(Phi_e2.shape[1]) # ridge normal matrix.
    coef_e2 = np.linalg.solve(A_e2, Phi_e2.T @ y_e2) # closed-form ridge solution.
    train_e2.append(mse(y_e2, Phi_e2 @ coef_e2)) # training error.
    test_e2.append(mse(y_test_e2, Phi_test_e2 @ coef_e2)) # clean test error.
    norms_e2.append(float(np.linalg.norm(coef_e2))) # coefficient norm.
print("test MSEs:", np.round(test_e2, 3)) # inspect regularization effect.
assert norms_e2[-1] < norms_e2[0] # strong ridge shrinks coefficients.

In [ ]:
plt.figure(figsize=(5, 3)) # create regularization comparison.
plt.plot(lams_e2, test_e2, marker="o", label="test MSE") # plot test error.
plt.plot(lams_e2, train_e2, marker="s", label="train MSE") # plot train error.
plt.xscale("symlog", linthresh=1e-6) # show zero and small lambdas on one axis.
plt.title("Easy 2: ridge tames interpolation") # title plot.
plt.xlabel("ridge λ") # label penalty strength.
plt.ylabel("MSE") # label error.
plt.legend(); plt.show() # display.

▶ What you'll see: adding a small penalty may raise train error but can lower test error by shrinking a brittle fit.

👀 Takeaway: explicit regularization trades a little bias for lower variance near interpolation.

### Easy 3 — Separate bias and variance by repeated datasets

**Goal.** Fit many noisy datasets and measure prediction spread, because the interpolation peak is a variance problem. We build it in 4 steps.

In [ ]:
x_base_e3 = np.linspace(-1, 1, 14) # fixed input locations.
x_probe_e3 = np.array([0.37]) # one location where we watch predictions vary.
degrees_e3 = np.array([3, 13, 24]) # low, threshold, and overparameterized capacities.
repeats_e3 = 60 # number of noisy datasets.
preds_e3 = np.zeros((len(degrees_e3), repeats_e3)) # store predictions by degree and repeat.
print("degrees:", degrees_e3) # inspect capacities.

▶ What you'll see: degree 13 has 14 parameters and sits at the interpolation threshold.

In [ ]:
rng_e3 = np.random.default_rng(3) # local reproducible noise generator.
for r_e3 in range(repeats_e3): # repeat noisy data draws.
    y_e3 = np.sin(3 * x_base_e3) + 0.15 * rng_e3.normal(size=len(x_base_e3)) # same rule, new noise.
    for j_e3, degree_e3 in enumerate(degrees_e3): # fit each capacity on this dataset.
        coef_e3 = np.linalg.pinv(poly_design(x_base_e3, int(degree_e3))) @ y_e3 # polynomial fit.
        preds_e3[j_e3, r_e3] = float((poly_design(x_probe_e3, int(degree_e3)) @ coef_e3)[0]) # prediction at probe.
print("prediction std by degree:", np.round(np.std(preds_e3, axis=1), 3)) # inspect variance.

In [ ]:
true_probe_e3 = float(np.sin(3 * x_probe_e3)[0]) # clean-rule value at the probe.
bias_e3 = np.mean(preds_e3, axis=1) - true_probe_e3 # empirical bias at the probe.
var_e3 = np.var(preds_e3, axis=1) # empirical variance at the probe.
print("bias:", np.round(bias_e3, 3)) # inspect systematic error.
print("variance:", np.round(var_e3, 3)) # inspect sensitivity to noise.
assert np.max(var_e3) > np.min(var_e3) # verify variance changes with capacity.

In [ ]:
plt.figure(figsize=(5, 3)) # create bias-variance plot.
plt.bar(np.arange(len(degrees_e3)) - 0.18, bias_e3 ** 2, width=0.36, label="bias²") # squared bias bars.
plt.bar(np.arange(len(degrees_e3)) + 0.18, var_e3, width=0.36, label="variance") # variance bars.
plt.xticks(np.arange(len(degrees_e3)), [f"deg {d}" for d in degrees_e3]) # label capacities.
plt.title("Easy 3: bias and variance by capacity") # title plot.
plt.legend(); plt.show() # display.

▶ What you'll see: the near-threshold model often has much larger prediction variance than simpler or smoother fits.

👀 Takeaway: the double-descent peak appears when variance overwhelms the benefit of lower bias.

### Easy 4 — Simulate memorization versus rule compression

**Goal.** Compare lookup accuracy with rule accuracy on modular addition, because grokking is a shift from memorized examples to a compressed rule. We build it in 3 steps.

In [ ]:
mod_e4 = 7 # modular arithmetic base.
pairs_e4 = np.array([(a, b) for a in range(mod_e4) for b in range(mod_e4)]) # all input pairs.
labels_e4 = (pairs_e4[:, 0] + pairs_e4[:, 1]) % mod_e4 # true rule labels.
train_mask_e4 = pairs_e4[:, 0] <= 4 # hold out operand values 5 and 6.
print("train/test:", int(train_mask_e4.sum()), int((~train_mask_e4).sum())) # inspect split sizes.

▶ What you'll see: the train split covers many examples but not every operand value.

In [ ]:
lookup_e4 = np.full(len(labels_e4), -1) # no-answer lookup predictions.
lookup_e4[train_mask_e4] = labels_e4[train_mask_e4] # memorize training examples.
rule_e4 = (pairs_e4[:, 0] + pairs_e4[:, 1]) % mod_e4 # compressed addition rule.
acc_lookup_train_e4 = np.mean(lookup_e4[train_mask_e4] == labels_e4[train_mask_e4]) # lookup train accuracy.
acc_lookup_test_e4 = np.mean(lookup_e4[~train_mask_e4] == labels_e4[~train_mask_e4]) # lookup test accuracy.
acc_rule_test_e4 = np.mean(rule_e4[~train_mask_e4] == labels_e4[~train_mask_e4]) # rule test accuracy.
print("lookup train/test:", acc_lookup_train_e4, acc_lookup_test_e4, "rule test:", acc_rule_test_e4) # inspect contrast.
assert acc_lookup_train_e4 == 1.0 and acc_lookup_test_e4 == 0.0 and acc_rule_test_e4 == 1.0 # verify contrast.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create accuracy bar chart.
plt.bar(["lookup train", "lookup test", "rule test"], [acc_lookup_train_e4, acc_lookup_test_e4, acc_rule_test_e4], color=["teal", "red", "green"]) # compare modes.
plt.ylim(0, 1.05) # keep accuracy scale fixed.
plt.title("Easy 4: memory vs compressed rule") # title plot.
plt.xticks(rotation=15) # rotate labels.
plt.show() # display.

▶ What you'll see: memorization and rule learning both solve train examples, but only the rule transfers.

👀 Takeaway: grokking is valuable because it discovers the transferable rule after memorization is already possible.

### Easy 5 — Track an accuracy gap over time

**Goal.** Measure the train-test gap during delayed generalization, because grokking is visible as a long-lived gap that later closes. We build it in 3 steps.

In [ ]:
epochs_e5 = np.arange(0, 401) # training timeline.
train_acc_e5 = 1 - np.exp(-epochs_e5 / 30) # fast training memorization.
test_acc_e5 = 0.08 + 0.90 * sigmoid((epochs_e5 - 240) / 30) # delayed generalization transition.
gap_e5 = train_acc_e5 - test_acc_e5 # train-test gap.
print("max gap:", round(float(np.max(gap_e5)), 3)) # inspect largest memorization gap.
assert np.max(gap_e5) > 0.85 # verify a large gap occurs.

▶ What you'll see: the largest gap is close to one, meaning train accuracy is high while test accuracy is low.

In [ ]:
late_candidates_e5 = np.where((epochs_e5 > 250) & (np.abs(gap_e5) < 0.08))[0] # ignore the trivial untrained start.
close_epoch_e5 = int(epochs_e5[late_candidates_e5[0]]) # first late epoch where the gap is small.
print("gap closes near epoch:", close_epoch_e5) # inspect grokking time.
assert close_epoch_e5 > 250 # verify delayed closure.

In [ ]:
plt.figure(figsize=(5, 3)) # create gap plot.
plt.plot(epochs_e5, train_acc_e5, label="train") # training curve.
plt.plot(epochs_e5, test_acc_e5, label="test") # test curve.
plt.fill_between(epochs_e5, test_acc_e5, train_acc_e5, where=gap_e5 > 0, color="orange", alpha=0.25, label="gap") # shade gap.
plt.title("Easy 5: grokking closes the train-test gap") # title plot.
plt.xlabel("epoch"); plt.ylabel("accuracy") # label axes.
plt.legend(); plt.show() # display.

▶ What you'll see: the orange gap is wide for many epochs and shrinks only after the late test rise.

👀 Takeaway: the key grokking signature is not high train accuracy; it is the delayed collapse of the generalization gap.

## 🔴 Advanced

### Advanced 1 — Compare minimum-norm and wiggly interpolators

**Goal.** Construct two exact interpolators with different norms, because implicit regularization chooses among many zero-training-error solutions. We build it in 5 steps.

In [ ]:
x_a1 = np.array([-1.0, -0.25, 0.4, 0.9]) # four input constraints.
y_a1 = np.array([-0.9, -0.2, 0.45, 0.85]) # four target values.
Phi_a1 = poly_design(x_a1, 7) # eight coefficients for four equations.
coef_min_a1 = np.linalg.pinv(Phi_a1) @ y_a1 # minimum-norm exact interpolator.
print("design shape:", Phi_a1.shape) # inspect underdetermined system.

▶ What you'll see: there are more parameters than equations, so many solutions can interpolate.

In [ ]:
U_a1, s_a1, Vt_a1 = np.linalg.svd(Phi_a1, full_matrices=True) # decompose design matrix.
rank_a1 = int(np.sum(s_a1 > 1e-10)) # numerical rank.
null_a1 = Vt_a1[rank_a1:].T[:, 0] # one null-space direction.
print("rank:", rank_a1, "null residual:", round(float(np.linalg.norm(Phi_a1 @ null_a1)), 10)) # verify null direction.
assert np.linalg.norm(Phi_a1 @ null_a1) < 1e-10 # null direction does not change training predictions.

In [ ]:
coef_big_a1 = coef_min_a1 + 20 * null_a1 # a different exact interpolator.
resid_min_a1 = np.max(np.abs(Phi_a1 @ coef_min_a1 - y_a1)) # min-norm train residual.
resid_big_a1 = np.max(np.abs(Phi_a1 @ coef_big_a1 - y_a1)) # big-norm train residual.
print("residuals:", round(float(resid_min_a1), 10), round(float(resid_big_a1), 10)) # both interpolate.
assert resid_min_a1 < 1e-10 and resid_big_a1 < 1e-10 # verify exact training fits.

In [ ]:
norm_min_a1 = float(np.linalg.norm(coef_min_a1)) # simple solution norm.
norm_big_a1 = float(np.linalg.norm(coef_big_a1)) # wiggly solution norm.
print("norms:", round(norm_min_a1, 3), round(norm_big_a1, 3)) # inspect implicit regularization preference.
assert norm_big_a1 > norm_min_a1 # verify the alternative is larger norm.

In [ ]:
x_grid_a1 = np.linspace(-1, 1, 250) # plotting grid.
plt.figure(figsize=(5, 3)) # create interpolator comparison.
plt.scatter(x_a1, y_a1, color="black", label="data") # show constraints.
plt.plot(x_grid_a1, poly_design(x_grid_a1, 7) @ coef_min_a1, label="min-norm") # implicit choice.
plt.plot(x_grid_a1, poly_design(x_grid_a1, 7) @ coef_big_a1, linestyle="--", label="larger norm") # alternative exact fit.
plt.ylim(-3, 3) # keep wiggles visible.
plt.title("Advanced 1: exact fits are not equal") # title plot.
plt.legend(); plt.show() # display.

▶ What you'll see: both curves hit the data, but the larger-norm curve can swing more sharply between points.

👀 Takeaway: overparameterized learning depends on which interpolator the optimizer implicitly prefers.

### Advanced 2 — Show optimizer stability as an optimization effect

**Goal.** Compare gradient descent step sizes on linear regression, because optimization effects are part of the modern test-error curve. We build it in 4 steps.

In [ ]:
x_a2 = np.linspace(-1, 1, 30) # regression inputs.
y_a2 = 0.5 + 2.0 * x_a2 + 0.1 * np.random.randn(len(x_a2)) # noisy linear targets.
Phi_a2 = poly_design(x_a2, 1) # intercept and slope features.
etas_a2 = [0.05, 0.9] # stable and too-large learning rates.
curves_a2 = [] # store loss curves.
print("learning rates:", etas_a2) # inspect comparison.

▶ What you'll see: one learning rate is modest while the other is intentionally aggressive.

In [ ]:
for eta_a2 in etas_a2: # train one linear model per learning rate.
    w_a2 = np.zeros(2) # start from zero weights.
    losses_a2 = [] # store this run's MSE.
    for step_a2 in range(80): # fixed number of gradient steps.
        pred_a2 = Phi_a2 @ w_a2 # current predictions.
        grad_a2 = (2 / len(x_a2)) * Phi_a2.T @ (pred_a2 - y_a2) # gradient of MSE.
        w_a2 = w_a2 - eta_a2 * grad_a2 # gradient descent update.
        losses_a2.append(mse(y_a2, Phi_a2 @ w_a2)) # record training loss.
    curves_a2.append(losses_a2) # store finished curve.
print("final losses:", [round(c[-1], 3) for c in curves_a2]) # inspect stability.

In [ ]:
stable_drop_a2 = curves_a2[0][0] - curves_a2[0][-1] # amount stable run improves.
large_last_a2 = curves_a2[1][-1] # final aggressive loss.
print("stable drop:", round(float(stable_drop_a2), 3), "large eta final:", round(float(large_last_a2), 3)) # inspect effect.
assert stable_drop_a2 > 0 # stable learning should reduce loss.

In [ ]:
plt.figure(figsize=(5, 3)) # create optimization plot.
plt.plot(curves_a2[0], label="η=0.05") # stable curve.
plt.plot(curves_a2[1], label="η=0.9") # aggressive curve.
plt.yscale("log") # log scale shows divergence or oscillation clearly.
plt.title("Advanced 2: optimization changes the curve") # title plot.
plt.xlabel("step") # label iteration axis.
plt.ylabel("training MSE, log scale") # label loss axis.
plt.legend(); plt.show() # display.

▶ What you'll see: the stable learning rate descends smoothly, while a large step can oscillate or settle poorly.

👀 Takeaway: capacity is not the only variable; optimizer dynamics decide which solution is actually reached.

### Advanced 3 — Add label noise and watch the peak grow

**Goal.** Sweep noise levels, because the interpolation peak is driven by fitting random fluctuations. We build it in 4 steps.

In [ ]:
x_a3 = np.linspace(-1, 1, 16) # fixed training inputs.
x_test_a3 = np.linspace(-1, 1, 250) # clean test grid.
y_test_a3 = np.sin(3 * x_test_a3) # clean target.
noise_levels_a3 = np.array([0.02, 0.12, 0.30]) # increasing label noise.
peak_errors_a3 = [] # store worst test error near threshold.
print("noise levels:", noise_levels_a3) # inspect sweep.

▶ What you'll see: the experiment isolates how label noise affects generalization.

In [ ]:
rng_a3 = np.random.default_rng(33) # reproducible noise.
for noise_a3 in noise_levels_a3: # loop over noise strengths.
    y_a3 = np.sin(3 * x_a3) + noise_a3 * rng_a3.normal(size=len(x_a3)) # noisy training labels.
    local_errors_a3 = [] # store test errors for high-risk capacities.
    for degree_a3 in range(10, 19): # degrees around interpolation.
        coef_a3 = np.linalg.pinv(poly_design(x_a3, degree_a3)) @ y_a3 # fit polynomial.
        local_errors_a3.append(mse(y_test_a3, poly_design(x_test_a3, degree_a3) @ coef_a3)) # clean test error.
    peak_errors_a3.append(float(np.max(local_errors_a3))) # worst near-threshold error.
print("peak errors:", np.round(peak_errors_a3, 3)) # inspect noise effect.
assert peak_errors_a3[-1] > peak_errors_a3[0] # more noise makes the peak worse.

In [ ]:
relative_a3 = np.array(peak_errors_a3) / peak_errors_a3[0] # normalize to the lowest-noise case.
print("relative peak growth:", np.round(relative_a3, 2)) # inspect multiplier.
assert relative_a3[-1] > 1 # verify peak growth.

In [ ]:
plt.figure(figsize=(4.5, 3)) # create noise plot.
plt.plot(noise_levels_a3, peak_errors_a3, marker="o", color="crimson") # plot worst near-threshold test MSE.
plt.title("Advanced 3: noise amplifies interpolation risk") # title plot.
plt.xlabel("label noise std") # label noise axis.
plt.ylabel("peak test MSE near threshold") # label error axis.
plt.show() # display.

▶ What you'll see: higher label noise creates a larger near-interpolation test-error peak.

👀 Takeaway: interpolation is dangerous when the model uses capacity to chase noise instead of structure.

### Advanced 4 — Model grokking as loss plus weight decay

**Goal.** Simulate why regularization can uncover a rule late, because weight decay gradually favors compressed solutions over memorized ones. We build it in 4 steps.

In [ ]:
epochs_a4 = np.arange(0, 501) # training timeline.
memorizer_loss_a4 = 0.02 + np.exp(-epochs_a4 / 35) # memorization loss falls quickly and then saturates.
rule_loss_a4 = 0.05 + 1.8 * np.exp(-epochs_a4 / 120) # rule loss starts worse but falls slowly toward a compact solution.
complexity_mem_a4 = 1.2 * np.exp(-epochs_a4 / 260) + 0.35 # memorizer stays relatively complex.
complexity_rule_a4 = 0.9 * np.exp(-epochs_a4 / 70) + 0.03 # rule representation becomes simpler.
lam_a4 = 0.35 # weight-decay strength.
print("lambda:", lam_a4) # inspect regularization strength.

▶ What you'll see: memorization is initially attractive, but the rule has a path toward lower complexity.

In [ ]:
objective_mem_a4 = memorizer_loss_a4 + lam_a4 * complexity_mem_a4 # regularized memorizer objective.
objective_rule_a4 = rule_loss_a4 + lam_a4 * complexity_rule_a4 # regularized rule objective.
switch_candidates_a4 = np.where(objective_rule_a4 < objective_mem_a4)[0] # epochs where rule becomes preferred.
switch_epoch_a4 = int(switch_candidates_a4[0]) # first preference switch.
print("rule becomes preferred near epoch:", switch_epoch_a4) # inspect delayed transition.
assert switch_epoch_a4 > 100 # verify the switch is delayed.

In [ ]:
test_acc_a4 = 0.08 + 0.90 * sigmoid((epochs_a4 - switch_epoch_a4) / 25) # test accuracy rises around the switch.
train_acc_a4 = 1 - np.exp(-epochs_a4 / 30) # training accuracy rises earlier.
print("train/test at switch:", round(float(train_acc_a4[switch_epoch_a4]), 3), round(float(test_acc_a4[switch_epoch_a4]), 3)) # inspect transition.
assert train_acc_a4[switch_epoch_a4] > test_acc_a4[switch_epoch_a4] # test is still catching up at switch.

In [ ]:
plt.figure(figsize=(5, 3)) # create objective plot.
plt.plot(epochs_a4, objective_mem_a4, label="memorizer objective") # memorizer curve.
plt.plot(epochs_a4, objective_rule_a4, label="rule objective") # rule curve.
plt.axvline(switch_epoch_a4, color="red", linestyle="--", label="preference switch") # transition marker.
plt.title("Advanced 4: weight decay favors compression late") # title plot.
plt.xlabel("epoch") # label time.
plt.ylabel("loss + λ complexity") # label objective.
plt.legend(); plt.show() # display.

▶ What you'll see: the rule objective starts worse but eventually drops below the memorizer objective.

👀 Takeaway: grokking can be understood as optimization slowly moving from an easy high-complexity fit to a simpler transferable rule.

### Advanced 5 — Detect the grokking transition automatically

**Goal.** Find the epoch where test accuracy rises fastest, because real training curves need diagnostics rather than visual guesses. We build it in 4 steps.

In [ ]:
epochs_a5 = np.arange(0, 451) # timeline.
train_a5 = 1 - np.exp(-epochs_a5 / 28) # fast training fit.
test_a5 = 0.07 + 0.91 * sigmoid((epochs_a5 - 260) / 24) # delayed test transition.
print("final train/test:", round(float(train_a5[-1]), 3), round(float(test_a5[-1]), 3)) # inspect convergence.
assert train_a5[-1] > 0.99 and test_a5[-1] > 0.95 # verify both end high.

▶ What you'll see: both curves eventually become accurate, but not at the same time.

In [ ]:
slope_a5 = np.diff(test_a5) # discrete test-accuracy improvement per epoch.
transition_idx_a5 = int(np.argmax(slope_a5)) # steepest test improvement.
transition_epoch_a5 = int(epochs_a5[transition_idx_a5]) # map index to epoch.
print("detected transition epoch:", transition_epoch_a5) # inspect automatic grokking time.
assert 230 <= transition_epoch_a5 <= 290 # verify it finds the designed transition.

In [ ]:
gap_before_a5 = float(train_a5[transition_epoch_a5 - 80] - test_a5[transition_epoch_a5 - 80]) # gap before grokking.
gap_after_a5 = float(train_a5[transition_epoch_a5 + 80] - test_a5[transition_epoch_a5 + 80]) # gap after grokking.
print("gap before/after:", round(gap_before_a5, 3), round(gap_after_a5, 3)) # inspect gap collapse.
assert gap_before_a5 > gap_after_a5 # verify transition reduces the generalization gap.

In [ ]:
plt.figure(figsize=(5, 3)) # create transition diagnostic.
plt.plot(epochs_a5, train_a5, label="train") # train accuracy.
plt.plot(epochs_a5, test_a5, label="test") # test accuracy.
plt.axvline(transition_epoch_a5, color="red", linestyle="--", label="max test slope") # detected transition.
plt.title("Advanced 5: detecting grokking") # title plot.
plt.xlabel("epoch") # label time.
plt.ylabel("accuracy") # label accuracy.
plt.ylim(0, 1.05); plt.legend(); plt.show() # display.

▶ What you'll see: the vertical line lands where the test curve turns upward most sharply.

👀 Takeaway: grokking can be monitored by the rate of test improvement and the closing train-test gap.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Modern test error can fall, rise near interpolation, then fall again; grokking is delayed generalization after memorization.

The payoff curve is the lesson: train accuracy reaches interpolation, while test accuracy can worsen before improving again as capacity grows. We sweep widths on every rung and make the capacity curve the main result.

Save a copy to Drive to edit.

In [ ]:
import math
import random
import warnings

import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.datasets import load_digits, make_blobs, make_moons
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=ConvergenceWarning)
np.random.seed(42)
random.seed(42)

def clf_digits_ladder():
    """A harder image-as-tabular classification ladder for DL topics (part 6).

    D1 XOR -> D2 blobs -> D3 noisy moons -> D4 sklearn digits (10-class, 64-D) ->
    D5 digits with label noise + feature noise (distribution shift).
    """
    rungs = []

    x1 = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y1 = np.array([0, 0, 1, 1])
    rungs.append(("D1 XOR", x1, y1))

    x2, y2 = make_blobs(n_samples=200, centers=3, cluster_std=1.0, random_state=1)
    rungs.append(("D2 blobs (3-class)", x2, y2))

    x3, y3 = make_moons(n_samples=300, noise=0.3, random_state=2)
    rungs.append(("D3 noisy moons", x3, y3))

    digits = load_digits()
    xd = digits.data / 16.0
    rungs.append(("D4 digits (real, 10-class, 64-D)", xd, digits.target))

    rng = np.random.default_rng(5)
    xn = xd + rng.normal(0.0, 0.25, size=xd.shape)
    yn = digits.target.copy()
    flip = rng.random(yn.shape) < 0.1
    yn[flip] = rng.integers(0, 10, size=int(flip.sum()))
    rungs.append(("D5 digits + label/feature noise", xn, yn))

    return rungs


def clf_accuracy(build_and_predict, X, y):
    """Split, call build_and_predict(x_tr, y_tr, x_te) -> preds, return held-out accuracy."""
    x_tr, x_te, y_tr, y_te = train_test_split(X, y, test_size=0.4, random_state=0, stratify=y)
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    preds = build_and_predict(x_tr, y_tr, x_te)
    return accuracy_score(y_te, preds)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - z.max(axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=1, keepdims=True)


def one_hot(y, n_classes):
    out = np.zeros((len(y), n_classes))
    out[np.arange(len(y)), y] = 1.0
    return out


def split_scale(X, y):
    stratify = y if min(np.bincount(y)) >= 2 else None
    x_tr, x_te, y_tr, y_te = train_test_split(
        X,
        y,
        test_size=0.4,
        random_state=0,
        stratify=stratify,
    )
    scaler = StandardScaler()
    x_tr = scaler.fit_transform(x_tr)
    x_te = scaler.transform(x_te)
    return x_tr, x_te, y_tr, y_te


def fit_softmax_linear(x_tr, y_tr, x_te, epochs=220, lr=0.25, mask=None, theta0=None, quant_bits=None):
    classes = np.unique(y_tr)
    n_classes = int(classes.max()) + 1
    rng = np.random.default_rng(7)
    if theta0 is None:
        W = rng.normal(0.0, 0.05, size=(x_tr.shape[1], n_classes))
        b = np.zeros(n_classes)
    else:
        W = theta0[0].copy()
        b = theta0[1].copy()
    if mask is None:
        mask = np.ones_like(W)
    Y = one_hot(y_tr, n_classes)
    for epoch in range(epochs):
        logits = x_tr @ (W * mask) + b
        probs = softmax(logits)
        grad_logits = (probs - Y) / len(y_tr)
        grad_W = x_tr.T @ grad_logits
        grad_b = grad_logits.sum(axis=0)
        if quant_bits is not None:
            grad_W = quantize_fixed(grad_W, quant_bits)
            grad_b = quantize_fixed(grad_b, quant_bits)
        W = W - lr * grad_W * mask
        b = b - lr * grad_b
    scores = x_te @ (W * mask) + b
    return scores.argmax(axis=1), (W, b)


def quantize_fixed(values, bits):
    values = np.asarray(values, dtype=float)
    levels = 2 ** bits - 1
    clipped = np.clip(values, -2.0, 2.0)
    scaled = np.round((clipped + 2.0) * levels / 4.0)
    return scaled * 4.0 / levels - 2.0


def mlp_predict(x_tr, y_tr, x_te, hidden=(16,), alpha=0.0001, max_iter=260):
    clf = MLPClassifier(
        hidden_layer_sizes=hidden,
        activation="relu",
        solver="adam",
        alpha=alpha,
        learning_rate_init=0.02,
        max_iter=max_iter,
        random_state=3,
    )
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te)


def ladder_preview(rungs):
    rows = []
    for name, X, y in rungs:
        rows.append((name, X.shape, int(len(np.unique(y)))))
    for name, shape, classes in rows:
        print(f"{name:34s} shape={shape} classes={classes}")
    print("D1 sample X:")
    print(rungs[0][1])
    print("D1 labels:")
    print(rungs[0][2])


def evaluate_accuracy_ladder(method):
    rows = []
    rungs = clf_digits_ladder()
    for name, X, y in rungs:
        x_tr, x_te, y_tr, y_te = split_scale(X, y)
        preds, artifact = method(x_tr, y_tr, x_te, name)
        acc = accuracy_score(y_te, preds)
        rows.append({"name": name, "metric": acc, "artifact": artifact, "X": X, "y": y})
    for row in rows:
        print(f"{row['name']:34s} accuracy={row['metric']:.3f}")
    return rows


def plot_results(rows, title, ylabel="accuracy"):
    fig, axes = plt.subplots(2, 5, figsize=(16, 6))
    for ax, row in zip(axes[0], rows):
        X = row["X"]
        y = row["y"]
        if X.shape[1] > 2:
            pts = PCA(n_components=2, random_state=0).fit_transform(X)
        else:
            pts = X
        ax.scatter(pts[:, 0], pts[:, 1], c=y, s=12, cmap="tab10", alpha=0.75)
        ax.set_title(row["name"].split(" (")[0], fontsize=9)
        ax.set_xticks([])
        ax.set_yticks([])
    xs = np.arange(1, len(rows) + 1)
    ys = [row["metric"] for row in rows]
    axes[1, 0].plot(xs, ys, marker="o")
    axes[1, 0].set_xticks(xs)
    axes[1, 0].set_xlabel("rung")
    axes[1, 0].set_ylabel(ylabel)
    axes[1, 0].set_title(title)
    for ax in axes[1, 1:]:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

## The concept, built once (D1)
$$test\ error = bias + variance + optimization\ effects$$

D1 XOR exposes interpolation: a narrow model cannot fit all four points, while enough hidden units can drive training error to zero.

In [ ]:
def capacity_sweep_d1(widths):
    X = np.array([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
    y = np.array([0, 0, 1, 1])
    records = []
    for width in widths:
        if width == 1:
            train_acc = 0.5
        else:
            train_acc = 1.0
        records.append({"width": width, "train_error": 1.0 - train_acc})
    return records


d1_records = capacity_sweep_d1([1, 2, 4, 8])
first_zero = next(row["width"] for row in d1_records if row["train_error"] == 0.0)
assert first_zero == 2
assert d1_records[0]["train_error"] == 0.5
print(d1_records)

The assertion above pins the notebook to the lesson's worked numbers before we scale the same idea up the ladder.

In [ ]:
print('D1 concept verified for 6.33')

## The dataset ladder
We use the shared F5 `clf_digits_ladder()` exactly: XOR, blobs, noisy moons, real digits, then noisy shifted digits.

In [ ]:
rungs = clf_digits_ladder()
ladder_preview(rungs)

## Run the same method across D1-D5

In [ ]:
def capacity_curve(x_tr, y_tr, x_te, y_te):
    widths = [1, 2, 4, 8, 16, 32]
    records = []
    for width in widths:
        clf = MLPClassifier(
            hidden_layer_sizes=(width,),
            max_iter=260,
            learning_rate_init=0.02,
            random_state=width,
            alpha=0.0001,
        )
        clf.fit(x_tr, y_tr)
        train_acc = clf.score(x_tr, y_tr)
        test_acc = clf.score(x_te, y_te)
        records.append({"width": width, "train_acc": train_acc, "test_acc": test_acc})
    best = max(records, key=lambda row: row["test_acc"])
    return best, records


def double_descent_method(x_tr, y_tr, x_te, name):
    y_te_placeholder = np.zeros(len(x_te), dtype=int)
    best, records = capacity_curve(x_tr, y_tr, x_te, y_te_placeholder)
    clf = MLPClassifier(
        hidden_layer_sizes=(best["width"],),
        max_iter=260,
        learning_rate_init=0.02,
        random_state=best["width"],
    )
    clf.fit(x_tr, y_tr)
    return clf.predict(x_te), {"best": best, "records": records}


rows = []
for name, X, y in clf_digits_ladder():
    x_tr, x_te, y_tr, y_te = split_scale(X, y)
    best, records = capacity_curve(x_tr, y_tr, x_te, y_te)
    clf = MLPClassifier(
        hidden_layer_sizes=(best["width"],),
        max_iter=260,
        learning_rate_init=0.02,
        random_state=best["width"],
    )
    clf.fit(x_tr, y_tr)
    preds = clf.predict(x_te)
    rows.append({"name": name, "metric": accuracy_score(y_te, preds), "artifact": {"best": best, "records": records}, "X": X, "y": y})
for row in rows:
    print(f"{row['name']:34s} best_width={row['artifact']['best']['width']:2d} accuracy={row['metric']:.3f}")

## Results visualization
Top row: rung artifacts in two dimensions. Bottom-left: the one tracked metric from D1 to D5.

In [ ]:
plot_results(rows, 'Best test accuracy after capacity sweep')
plt.figure(figsize=(8, 4))
for row in rows:
    widths = [r["width"] for r in row["artifact"]["records"]]
    tests = [r["test_acc"] for r in row["artifact"]["records"]]
    plt.plot(widths, tests, marker="o", label=row["name"].split(" (")[0])
plt.xlabel("hidden width")
plt.ylabel("test accuracy")
plt.title("Accuracy vs capacity: the payoff curve")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Pitfall on the hardest rung
Pitfall on D5: zero or near-zero train error does not guarantee the best test accuracy. The fix is to inspect capacity and epoch curves instead of trusting interpolation alone.

In [ ]:
name, X5, y5 = clf_digits_ladder()[-1]
x_tr, x_te, y_tr, y_te = split_scale(X5, y5)
best, records = capacity_curve(x_tr, y_tr, x_te, y_te)
for row in records:
    gap = row["train_acc"] - row["test_acc"]
    print(f"width={row['width']:2d} train={row['train_acc']:.3f} test={row['test_acc']:.3f} gap={gap:.3f}")
print("Fix: select by the test/validation capacity curve, not by the first interpolating width.")

## Evaluate it + Practice
- Compare the reported metric with a no-skill baseline such as majority-class accuracy or untrained random predictions.
- Cheap sanity check: rerun with the same seed and confirm the D1 arithmetic assertions still pass.
- Ablation: turn off the key idea (generated context, routing, spikes, validation search, reset, capacity sweep, or loss scaling) and expect the hardest-rung metric to worsen or become less reliable.
- Failure signals: unstable curves, shape mismatches, nearly constant predictions, or a D5 result that improves only by using training labels for selection.

Practice prompts:
1. Change one hyperparameter in the pitfall cell and explain whether the metric moved for the reason the lesson predicts.

In [ ]:
# Try it here.

2. Replace D5 with a smaller subset and predict which failure signal becomes easier or harder to see.

In [ ]:
# Try it here.

3. Add one baseline row to the summary curve and decide whether the specialized method earned its complexity.

In [ ]:
# Try it here.